In [ ]:
docker run -d --name neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password123 -v neo4j_data:/data neo4j:latest

In [3]:
from neo4j import GraphDatabase
URI = "bolt://localhost:7687" 
USERNAME = "neo4j"
PASSWORD = "password123"
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j successfully.")

Connected to Neo4j successfully.


In [5]:
with driver.session() as session:
    session.run("""
        CREATE CONSTRAINT customer_unique IF NOT EXISTS
        FOR (c:Customer)
        REQUIRE c.customer_id IS UNIQUE
    """)

    session.run("""
        CREATE CONSTRAINT device_unique IF NOT EXISTS
        FOR (d:Device)
        REQUIRE d.device_id IS UNIQUE
    """)

    session.run("""
        CREATE CONSTRAINT merchant_unique IF NOT EXISTS
        FOR (m:Merchant)
        REQUIRE m.merchant_id IS UNIQUE
    """)

print("Constraints created!")

Constraints created!


In [5]:
import pandas as pd
import glob
import os

DATA_PATH = r"E:\euron-super30\supabase_db_kit\data\tables"

def load_folder(folder):
    files = glob.glob(os.path.join(DATA_PATH, folder, "**", "*.csv*"), recursive=True)
    print(f"{folder}: {len(files)} files")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

tx = load_folder("transactions")
labels = load_folder("labels")
print("transactions:", len(tx), "| labels:", len(labels))

transactions: 23 files
labels: 23 files
transactions: 110000 | labels: 110000


In [7]:
print("transactions:", tx.columns.tolist())
print("labels:", labels.columns.tolist())

transactions: ['transaction_id', 'customer_id', 'merchant_id', 'device_id', 'amount_paise', 'transaction_amount', 'payment_channel', 'merchant_category', 'transaction_time', 'customer_age', 'customer_location', 'merchant_location', 'account_age', 'previous_transactions', 'average_transaction_value', 'failed_transactions', 'split', 'is_synthetic']
labels: ['transaction_id', 'fraud_label', 'label_source', 'label_available_at', 'split', 'eligible_at_next_split_start', 'is_synthetic']


In [8]:
#checkc the splits and date range
print(tx["split"].value_counts())

train_tx = tx[tx["split"] == "train"]
print("train rows:", len(train_tx))
print("train time range:", train_tx["transaction_time"].min(), "to", train_tx["transaction_time"].max())

split
train         72050
validation    14061
holdout       13889
warmup        10000
Name: count, dtype: int64
train rows: 72050
train time range: 2024-12-31T19:07:56.971+00:00 to 2025-10-31T18:16:30.969+00:00


In [9]:
needed = ["transaction_id", "customer_id", "device_id", "merchant_id",
          "transaction_amount", "transaction_time"]
print(train_tx[needed].isna().sum())

transaction_id          0
customer_id             0
device_id             858
merchant_id             0
transaction_amount      0
transaction_time        0
dtype: int64


In [10]:
#join labels and remove missing values
labels_train = labels[labels["split"] == "train"][["transaction_id", "fraud_label"]]

full = (
    train_tx.merge(labels_train, on="transaction_id", how="inner")
    .dropna(subset=["transaction_id", "customer_id", "device_id", "merchant_id",
                    "transaction_amount", "transaction_time", "fraud_label"])
    .drop_duplicates(subset=["transaction_id"])
)

print("clean train rows:", len(full))
print("fraud labels:", full["fraud_label"].value_counts().to_dict())

dev_customers = full.groupby("device_id")["customer_id"].nunique()
print("\ncustomers per device (whole train split):")
print(dev_customers.value_counts().sort_index())

clean train rows: 71192
fraud labels: {0: 70758, 1: 434}

customers per device (whole train split):
customer_id
1     4384
2        6
3       15
4       23
5       16
6       19
7       13
8        9
9        8
10       7
12       2
14       1
Name: count, dtype: int64


In [11]:
#build the 5,000-payment sample
shared = dev_customers[dev_customers >= 3].index
print("devices with 3+ customers:", len(shared))

shared_rows = full[full["device_id"].isin(shared)]
rest = full[~full["device_id"].isin(shared)]

filler = rest.sample(n=max(0, 5000 - len(shared_rows)), random_state=42)
sample = pd.concat([shared_rows, filler]).drop_duplicates(subset=["transaction_id"])

print("sample rows:", len(sample), "| from shared devices:", len(shared_rows))
print("time range:", sample["transaction_time"].min(), "to", sample["transaction_time"].max())

devices with 3+ customers: 113
sample rows: 5000 | from shared devices: 4583
time range: 2024-12-31T19:07:56.971+00:00 to 2025-10-31T18:16:30.969+00:00


In [12]:
rows = []
for r in sample.itertuples(index=False):
    rows.append({
        "transaction_id": str(r.transaction_id),
        "customer_id": str(r.customer_id),
        "device_id": str(r.device_id),
        "merchant_id": str(r.merchant_id),
        "amount": float(r.transaction_amount),
        "time": str(r.transaction_time),
        "fraud_label": int(r.fraud_label),
    })

print("rows prepared:", len(rows))
print("first row:", rows[0])

rows prepared: 5000
first row: {'transaction_id': 'TX_0000010001', 'customer_id': 'C_8403fb2e903349ff', 'device_id': 'D_0a7613e59c6db90f', 'merchant_id': 'M_5fde3bb6a98424ab', 'amount': 12446.04, 'time': '2024-12-31T19:07:56.971+00:00', 'fraud_label': 0}


In [13]:
#create the nodes
NODES = {
    "Customer": ("customer_id", "customer_id"),
    "Device": ("device_id", "device_id"),
    "Merchant": ("merchant_id", "merchant_id"),
}

with driver.session() as session:
    for label, (prop, key) in NODES.items():
        values = list({r[key] for r in rows})
        for i in range(0, len(values), 500):
            session.run(
                f"UNWIND $vals AS v MERGE (n:{label} {{{prop}: v}})",
                vals=values[i:i + 500],
            ).consume()
        print(label, "nodes merged:", len(values))

Customer nodes merged: 859
Device nodes merged: 502
Merchant nodes merged: 173


In [14]:
#connect customers to devices (USED_DEVICE)
USED = """
UNWIND $rows AS row
MATCH (c:Customer {customer_id: row.customer_id})
MATCH (d:Device {device_id: row.device_id})
MERGE (c)-[:USED_DEVICE]->(d)
"""

with driver.session() as session:
    for i in range(0, len(rows), 500):
        session.run(USED, rows=rows[i:i + 500]).consume()
print("USED_DEVICE done")

USED_DEVICE done


In [15]:
#create the payments (PAID)
PAID = """
UNWIND $rows AS row
MATCH (c:Customer {customer_id: row.customer_id})
MATCH (m:Merchant {merchant_id: row.merchant_id})
MERGE (c)-[p:PAID {transaction_id: row.transaction_id}]->(m)
SET p.amount = row.amount,
    p.time = datetime(row.time),
    p.fraud_label = row.fraud_label,
    p.device_id = row.device_id
"""

with driver.session() as session:
    for i in range(0, len(rows), 500):
        session.run(PAID, rows=rows[i:i + 500]).consume()
        print("PAID loaded", min(i + 500, len(rows)))
print("PAID done")

PAID loaded 500
PAID loaded 1000
PAID loaded 1500
PAID loaded 2000
PAID loaded 2500
PAID loaded 3000
PAID loaded 3500
PAID loaded 4000
PAID loaded 4500
PAID loaded 5000
PAID done


In [6]:
#device shared by 3 or more customer in last 7 days
query = """
MATCH (d:Device {device_id: $device_id})
      <-[:USED_DEVICE]-(c:Customer)
      -[p:PAID]->(m:Merchant)

WHERE p.device_id = d.device_id
  AND p.time >= datetime($window_start)
  AND p.time < datetime($window_start) + duration({days: 7})

RETURN
    d.device_id AS device_id,
    c.customer_id AS customer_id,
    COUNT(p) AS transaction_count,
    COUNT(DISTINCT m) AS merchant_count,
    SUM(p.amount) AS total_amount,
    COLLECT(DISTINCT p.fraud_label) AS fraud_labels

ORDER BY transaction_count DESC
"""

parameters = {
    "device_id": "D_c48c331b4ed19bce",
    "window_start": "2025-02-21T21:52:44Z"
}

with driver.session() as session:
    result = session.run(query, parameters)

    records = [record.data() for record in result]

df = pd.DataFrame(records)

display(df)

,device_id,customer_id,transaction_count,merchant_count,total_amount,fraud_labels
0,D_c48c331b4ed19bce,C_a3fef1d1bb96c6fe,8,1,4367.88,"[0, 1]"
1,D_c48c331b4ed19bce,C_31b398af779a1a10,7,1,3737.51,[0]
2,D_c48c331b4ed19bce,C_2bd85847476a6871,7,1,12493.41,"[0, 1]"


In [18]:
#Add fraud investigation logic in Python
def classify_device(row):
    if len(row["fraud_labels"]) > 1 or 1 in row["fraud_labels"]:
        return "Suspicious"
    return "No fraud label detected"

df["risk_status"] = df.apply(classify_device, axis=1)

display(df)

,device_id,customer_id,transaction_count,merchant_count,total_amount,fraud_labels,risk_status
0,D_c48c331b4ed19bce,C_a3fef1d1bb96c6fe,8,1,4367.88,"[0, 1]",Suspicious
1,D_c48c331b4ed19bce,C_31b398af779a1a10,7,1,3737.51,[0],No fraud label detected
2,D_c48c331b4ed19bce,C_2bd85847476a6871,7,1,12493.41,"[0, 1]",Suspicious


In [19]:
#Calculate overall device statistics
print("Total customers:", df["customer_id"].nunique())

print("Total transactions:", df["transaction_count"].sum())

print("Total transaction amount:", round(df["total_amount"].sum(), 2))

print(
    "Customers with fraud labels:",
    df["fraud_labels"].apply(lambda x: 1 in x).sum()
)

Total customers: 3
Total transactions: 22
Total transaction amount: 20598.8
Customers with fraud labels: 2


In [ ]:
#fraud record
fraud_query = """
MATCH (c:Customer)-[p:PAID]->(m:Merchant)
WHERE p.device_id = $device_id
  AND p.time >= datetime($window_start)
  AND p.time < datetime($window_start) + duration({days: 7})
  AND p.fraud_label = 1
RETURN c.customer_id AS customer_id,
       m.merchant_id AS merchant_id,
       p.amount AS amount,
       p.time AS time
ORDER BY time
"""

with driver.session() as session:
    fraud_records = [r.data() for r in session.run(fraud_query, parameters)]

display(pd.DataFrame(fraud_records))

,customer_id,merchant_id,amount,time
0,C_2bd85847476a6871,M_8613b75179e11c81,7476.49,2025-02-26T18:54:39.976000000+00:00
1,C_a3fef1d1bb96c6fe,M_73fa75978cd80349,1727.85,2025-02-26T22:06:34.649000000+00:00


In [10]:
count_query = """
MATCH (fc:Customer)-[:PAID {fraud_label: 1}]->(:Merchant)
WITH DISTINCT fc
MATCH (fc)-[:USED_DEVICE|PAID*1..2]-(near:Customer)
WHERE near <> fc
RETURN count(DISTINCT near) AS total_nearby
"""

with driver.session() as session:
    total = session.run(count_query).single()["total_nearby"]

print("total customers within 2 hops of a fraud customer:", total)

total customers within 2 hops of a fraud customer: 707


In [11]:
split_query = """
MATCH (fc:Customer)-[:PAID {fraud_label: 1}]->(:Merchant)
WITH DISTINCT fc

// 1 hop: shares a device directly
OPTIONAL MATCH (fc)-[:USED_DEVICE]->(:Device)<-[:USED_DEVICE]-(d_near:Customer)
WHERE d_near <> fc
WITH fc, collect(DISTINCT d_near) AS via_device

// 2 hops: shares a merchant
OPTIONAL MATCH (fc)-[:PAID]->(:Merchant)<-[:PAID]-(m_near:Customer)
WHERE m_near <> fc
WITH via_device, collect(DISTINCT m_near) AS via_merchant
WITH via_device + via_merchant AS everyone
UNWIND everyone AS n
RETURN count(DISTINCT n) AS all_nearby
"""

device_query = """
MATCH (fc:Customer)-[:PAID {fraud_label: 1}]->(:Merchant)
MATCH (fc)-[:USED_DEVICE]->(:Device)<-[:USED_DEVICE]-(near:Customer)
WHERE near <> fc
RETURN count(DISTINCT near) AS via_shared_device
"""

merchant_query = """
MATCH (fc:Customer)-[:PAID {fraud_label: 1}]->(:Merchant)
MATCH (fc)-[:PAID]->(:Merchant)<-[:PAID]-(near:Customer)
WHERE near <> fc
RETURN count(DISTINCT near) AS via_shared_merchant
"""

with driver.session() as session:
    dev = session.run(device_query).single()["via_shared_device"]
    mer = session.run(merchant_query).single()["via_shared_merchant"]

print("linked through a shared device:", dev)
print("linked through a shared merchant:", mer)

linked through a shared device: 541
linked through a shared merchant: 705


In [12]:
tight_query = """
MATCH (fc:Customer)-[fp:PAID {fraud_label: 1}]->(:Merchant)
MATCH (near:Customer)-[np:PAID]->(:Merchant)
WHERE near <> fc
  AND np.device_id = fp.device_id
  AND np.time >= fp.time - duration({days: 7})
  AND np.time <= fp.time + duration({days: 7})
RETURN count(DISTINCT near) AS near_on_fraud_device_7_days
"""

with driver.session() as session:
    tight = session.run(tight_query).single()["near_on_fraud_device_7_days"]

print("customers who paid on a fraud device within 7 days:", tight)

customers who paid on a fraud device within 7 days: 66


In [13]:
check = """
MATCH (fc:Customer)-[fp:PAID {fraud_label: 1}]->(:Merchant)
MATCH (near:Customer {customer_id: 'C_31b398af779a1a10'})-[np:PAID]->(:Merchant)
WHERE np.device_id = fp.device_id
  AND np.time >= fp.time - duration({days: 7})
  AND np.time <= fp.time + duration({days: 7})
RETURN near.customer_id AS customer, count(np) AS payments_near_fraud, collect(DISTINCT fc.customer_id) AS fraud_customers
"""

with driver.session() as session:
    display(pd.DataFrame([r.data() for r in session.run(check)]))

,customer,payments_near_fraud,fraud_customers
0,C_31b398af779a1a10,14,"[C_2bd85847476a6871, C_a3fef1d1bb96c6fe]"


In [14]:
#the merchants most often paid from shared devices.

query3 = """
MATCH (c:Customer)-[p:PAID]->(:Merchant)
WITH p.device_id AS device_id, collect(DISTINCT c.customer_id) AS customers
WHERE size(customers) >= 3
WITH collect(device_id) AS shared_devices
MATCH (c:Customer)-[p:PAID]->(m:Merchant)
WHERE p.device_id IN shared_devices
RETURN m.merchant_id AS merchant,
       count(p) AS payments_from_shared_devices,
       count(DISTINCT c) AS customers,
       sum(CASE WHEN p.fraud_label = 1 THEN 1 ELSE 0 END) AS fraud_payments
ORDER BY payments_from_shared_devices DESC
LIMIT 10
"""

with driver.session() as session:
    df3 = pd.DataFrame([r.data() for r in session.run(query3)])

display(df3)

,merchant,payments_from_shared_devices,customers,fraud_payments
0,M_15b31ab891d0241f,142,21,8
1,M_e56a56c8b785ae7c,123,17,7
2,M_8b7834813bfa6758,117,18,5
3,M_348a9e09345c7477,116,18,5
4,M_d945de86dbb6eb24,114,16,4
5,M_3e6d613a0de619b9,112,17,6
6,M_5f9ec56b007734f2,111,15,11
7,M_aad2c9114ea977bf,111,17,7
8,M_8613b75179e11c81,106,15,8
9,M_93f8de771fbee5e9,102,14,4
